## Modules & functions import

In [1]:
import os
import sys
from pprint import pprint

In [2]:
sys.path.append("../common_key_functions/")

In [3]:
from constants_configs import MODELS_CONFIGS, IMAGENET_CONSTANTS
from invert_batch_imagenet_funcs import val_model_on_files, \
    get_grid_images_paths, split_comparison_val_image, \
    get_labels_for_val_from_batch, get_labels_for_val_from_names, \
    sort_labels_by_images

## Specific pipeline functions

In [12]:
def val_inversion(networks_names: list[str],
                  orig_images_dirs: list[str],
                  recon_images_dirs: list[str],
                  labels_true_all: list[list | dict],
                  num_workers: int | None = 8,
                  print_details: bool | None = True,
                  top_k_preds: int | None = 1) -> dict[ str:float ]:
    """
    Run validation on original and reconstructed images for a list of networks.

    For each network, validates both the original images and the corresponding
    reconstructions, printing classification accuracy and optional top-k predictions.

    Args:
        networks_names:    List of network architecture names.
        orig_images_dirs:  List of directories containing original images (one per network).
        recon_images_dirs: List of directories containing reconstructed images (one per network).
        labels_true_all:   List of ground truth label lists (one per network).
        num_workers:       Number of DataLoader workers.
        print_details:     Whether to print per-image prediction details.
        top_k_preds:       Number of top predictions to display when `print_details` is True.
        
    Returns:
        Classification accuracy & number of correctly classified images for each network
    """
    networks_val_results = {}
    
    entities_to_iterate = zip(networks_names, orig_images_dirs, recon_images_dirs, labels_true_all) \
        if isinstance(labels_true_all[0], list) else \
                          zip(
                              networks_names, 
                              orig_images_dirs, 
                              recon_images_dirs, 
                              sort_labels_by_images(labels_true_all, orig_images_dirs)
                          )    
    for network_name, orig_images_dir, recon_images_dir, labels_true in entities_to_iterate: 
        classes_ids_n = len(set(labels_true))
        print(f"\n\nNetwork: {network_name}")
        
        print("\n    Original images validation:\n")
        acc_orig, correct_orig, total_orig = val_model_on_files(
            network_name,
            orig_images_dir,
            labels_true,
            max(MODELS_CONFIGS[network_name]["batch_size"], classes_ids_n),
            num_workers,
            print_details,
            top_k_preds
        )
        
        print("\n    Reconstructed images validation:\n")   
        acc_recon, correct_recon, total_recon = val_model_on_files(
            network_name,
            recon_images_dir,
            labels_true,
            max(MODELS_CONFIGS[network_name]["batch_size"], classes_ids_n),
            num_workers,
            print_details,
            top_k_preds
        )
        
        networks_val_results[network_name] = {
            "orig_accuracy" :  acc_orig * 100,
            "orig_correct" :   correct_orig,
            "recon_accuracy" : acc_recon * 100,
            "recon_correct" :  correct_recon,
            "total_images" :   total_orig,        
        }
        
    return networks_val_results

## Validation

### Constants setting

In [5]:
dataset_dir_imagenet = "/media/user/Hitachi/ILSVRC/Data/CLS-LOC"
dataset_name = "ImageNet"

classes_ids = "1, 10, 100, 999"
classes_ids_int = [ int(class_id.strip()) for class_id in classes_ids.split(",") ]
classes_ids_n = len(classes_ids_int)

networks_names = ["regnet_x_3_2", "regnet_x_16", "resnet50"] # MODELS_CONFIGS.keys()
networks_batch_sizes = {
    "regnet_x_3_2" : [8, 4, 2, 1],
    "regnet_x_16" :  [2, 1],
    "resnet50":      [8, 4, 2, 1]
}
networks_fake_batch_sizes = {
    network_name : classes_ids_n - 1
    for network_name in networks_names
}
select_best_n = 3
val_images_dir = "inversion_images_val_diff_batch_size"

### Data preparation from training

During training cycle the best samples of original images with their reconstructed versions are saved in a grid of the following format:

In [ ]:
# for usual training results with name "best_orig_vs_recon_<classes_ids>.png"
# networks_grid_images_paths = get_grid_images_paths(
    # networks_names, 
    # val_images_dir, 
    # classes_ids_int
# )

# for renamed training results
networks_grid_images_paths = get_grid_images_paths(
    networks_names,
    val_images_dir,
    classes_ids_int,
    networks_fake_batch_sizes
)
pprint(networks_grid_images_paths, indent=4)

{   'regnet_x_16': [   'inversion_images_val_diff_batch_size/regnet_x_16/bs_1_0001.png',
                       'inversion_images_val_diff_batch_size/regnet_x_16/bs_1_0999.png',
                       'inversion_images_val_diff_batch_size/regnet_x_16/bs_2_0001_0010.png',
                       'inversion_images_val_diff_batch_size/regnet_x_16/bs_1_0010.png',
                       'inversion_images_val_diff_batch_size/regnet_x_16/bs_2_0100_0999.png',
                       'inversion_images_val_diff_batch_size/regnet_x_16/bs_1_0100.png'],
    'regnet_x_3_2': [   'inversion_images_val_diff_batch_size/regnet_x_3_2/bs_8.png',
                        'inversion_images_val_diff_batch_size/regnet_x_3_2/bs_1_0001.png',
                        'inversion_images_val_diff_batch_size/regnet_x_3_2/bs_1_0999.png',
                        'inversion_images_val_diff_batch_size/regnet_x_3_2/bs_2_0001_0010.png',
                        'inversion_images_val_diff_batch_size/regnet_x_3_2/bs_1_0010.png',


In [ ]:
for network_name, grid_images_paths in networks_grid_images_paths.items():
    for grid_image_i, grid_image_path in enumerate(grid_images_paths):
        grid_image_batch_size = int(os.path.splitext(os.path.basename(grid_image_path))[0].split("_")[1])
        split_comparison_val_image(
            grid_image_path,
            os.path.dirname(grid_image_path),
            IMAGENET_CONSTANTS["size_center_crop"] + 2,     # empirically adjusted padding
            # MODELS_CONFIGS[network_name]["batch_size"],   # for usual validation
            batch_size=grid_image_batch_size,
            select_best_n=select_best_n,
            sample_number_shift=grid_image_i
        )

### Inversion validation

In [13]:
# for normal validation
# true_labels = [
#     get_labels_for_val_from_batch(
#         classes_ids_int, 
#         max(MODELS_CONFIGS[network_name]["batch_size"], classes_ids_n), 
#         select_best_n
#     )
#     for network_name in networks_names
# ]

# for custon validation
true_labels = [ 
    {
        image_name: int(label)
        for image_name, label in get_labels_for_val_from_names(
            os.path.join(val_images_dir, network_name, "origs")
        ).items()
    }
    for network_name in networks_names 
]

networks_val_results = val_inversion(
    networks_names,
    [ os.path.join(val_images_dir, network_name, "origs")  for network_name in networks_names ],
    [ os.path.join(val_images_dir, network_name, "recons") for network_name in networks_names ],
    true_labels,
    top_k_preds=5
)



Network: regnet_x_3_2

    Original images validation:

File: bs_1_0001_0001.png
    True class: 1
    Top-5 predictions: ['class 1 (0.96091014)', 'class 130 (0.00059600)', 'class 58 (0.00058600)', 'class 115 (0.00038019)', 'class 100 (0.00029179)']
File: bs_1_0010_0004.png
    True class: 10
    Top-5 predictions: ['class 10 (0.76817125)', 'class 761 (0.00125180)', 'class 13 (0.00118949)', 'class 86 (0.00100366)', 'class 371 (0.00091510)']
File: bs_1_0100_0006.png
    True class: 100
    Top-5 predictions: ['class 100 (0.63842756)', 'class 99 (0.02223999)', 'class 146 (0.01442633)', 'class 98 (0.01360639)', 'class 128 (0.00783808)']
File: bs_1_0999_0002.png
    True class: 999
    Top-5 predictions: ['class 999 (0.83215028)', 'class 731 (0.04101690)', 'class 191 (0.00643189)', 'class 861 (0.00457963)', 'class 179 (0.00354392)']
File: bs_2_0001_0003.png
    True class: 1
    Top-5 predictions: ['class 1 (0.96091014)', 'class 130 (0.00059600)', 'class 58 (0.00058600)', 'class 115 (0.0

In [14]:
pprint(networks_val_results, indent=4)

{   'regnet_x_16': {   'orig_accuracy': 100.0,
                       'orig_correct': 8,
                       'recon_accuracy': 50.0,
                       'recon_correct': 4,
                       'total_images': 8},
    'regnet_x_3_2': {   'orig_accuracy': 100.0,
                        'orig_correct': 14,
                        'recon_accuracy': 28.57142857142857,
                        'recon_correct': 4,
                        'total_images': 14},
    'resnet50': {   'orig_accuracy': 100.0,
                    'orig_correct': 14,
                    'recon_accuracy': 57.14285714285714,
                    'recon_correct': 8,
                    'total_images': 14}}
